# Gen 21+ Evolutionary Autoresearch: Unconstrained 17-Mul $\mathrm{Perm}_4$ DAGs (GPU Levenberg–Marquardt)

Runs `autoresearch/perm4_gpu_lm_autoresearch.py` on Google Colab GPU (Tesla T4 / L4 / A100) using:
- **Phase 1:** Batched PyTorch `AdamW` + cosine annealing across parallel restarts
- **Phase 2:** Batched GPU Levenberg–Marquardt (`torch.func.vmap` + `jacrev` + `linalg.solve`)
- **Anti-cheat:** Dual 61-bit Mersenne prime + `TrackedFieldElement` + SymPy $\mathbb{Q}[A]$ verification (`dag_verification.py`)

**Runtime:** Select **Runtime → Change runtime type → T4 GPU** (or A100).

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch, sympy, scipy; print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

In [ ]:
%%bash
set -euo pipefail
if [ ! -d matrix_permanent_circuit_discovery ]; then
  git clone https://github.com/3DBarath/matrix_permanent_circuit_discovery.git
fi
cd matrix_permanent_circuit_discovery
git pull --ff-only || true
ls -la autoresearch/

In [ ]:
%%bash
set -euo pipefail
cd matrix_permanent_circuit_discovery
# Quick 1-generation smoke test to verify GPU AdamW + batched Levenberg-Marquardt
PYTHONPATH=autoresearch python3 autoresearch/perm4_gpu_lm_autoresearch.py \
  --generations 1 --pop-size 2 --restarts 4 --torch-steps 100 \
  --log autoresearch/evolution_log_smoke.jsonl

In [ ]:
%%bash
set -euo pipefail
cd matrix_permanent_circuit_discovery
# Full GPU evolutionary autoresearch run
PYTHONPATH=autoresearch python3 autoresearch/perm4_gpu_lm_autoresearch.py \
  --generations 20 --pop-size 8 --restarts 64 --torch-steps 400 \
  --log autoresearch/evolution_log_gpu.jsonl

In [ ]:
import json
from pathlib import Path

log_path = Path("matrix_permanent_circuit_discovery/autoresearch/evolution_log_gpu.jsonl")
rows = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
rows.sort(key=lambda r: r["metrics"]["val_mse"])

print(f"Total evaluations: {len(rows)}")
for r in rows[:10]:
    m, g = r["metrics"], r["genome"]
    print(
        f"gen={r['gen']:2d} val_mse={m['val_mse']:.3e} rel_rmse={m.get('val_rel_rmse', float('nan')):.4f} "
        f"exact={m['verified_exact']} fam={g['family']} split={g['split']} warm_drop={g['warm_drop_gate']}"
    )

try:
    from google.colab import files
    files.download(str(log_path))
except ImportError:
    pass